# Hands-on 13: Ekspor Dataset ke CSV dan Publish ke MQTT Simulator

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 4.

Dataset klasifikasi (`data/synthetic/room_classification_dataset.csv`) dipublikasikan
baris per baris ke topik `iotlab/room01/telemetry`, diverifikasi oleh subscriber,
dan varian multi-topic mengirim warning/fault juga ke `iotlab/room01/alert`.

## Tujuan

Pada latihan ini, pembaca akan:

- membaca dataset sintetis dari CSV,
- mengubah setiap baris menjadi payload JSON,
- lalu mengirimkannya ke broker MQTT sebagai simulator sensor.

Ini sangat penting karena di sinilah data sintetis benar-benar dipakai untuk menggerakkan pipeline IoT, bukan hanya disimpan sebagai file statis. Pendekatan ini sangat cocok dengan arsitektur buku Anda yang menekankan MQTT Mosquitto, subscriber Python, storage, dan dashboard.

Selain itu, Internet of Things and Data Analytics Handbook menjelaskan bahwa untuk banyak use case IoT, pola publish-subscribe sangat cocok sebagai pelengkap REST karena lebih sesuai dengan aplikasi yang event-driven.

## Mengapa langkah ini sangat penting?

Karena pada titik ini pembaca sudah bisa:

- membuat data sintetis,
- menyimpannya ke CSV,
- mengubahnya menjadi aliran telemetry,
- dan memakainya untuk menguji sistem end-to-end.

Ini sangat dekat dengan praktik nyata di edge/gateway Linux, yang menurut referensi memang cocok dipakai untuk pemrosesan lokal, translasi protokol, dan orkestrasi data sebelum ke sistem analitik yang lebih tinggi.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /downloads/iot-ml-book


In [2]:
import subprocess, sys, time

PYTHON = sys.executable
LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def start_process(args, stdout_file):
    f = open(stdout_file, "w", encoding="utf-8")
    proc = subprocess.Popen(args, stdout=f, stderr=subprocess.STDOUT)
    return proc, f

broker_proc, broker_f = start_process(
    [PYTHON, "scripts/mqtt/run_broker.py"], LOG_DIR / "broker_nb13.log"
)
time.sleep(3)
print("[INFO] Broker berjalan:", broker_proc.poll() is None)

[INFO] Broker berjalan: True


## 1. Publish CSV ke MQTT + Verifikasi Subscriber

In [3]:
# Subscriber verifikasi berjalan dulu, lalu publisher mengirim 20 baris.
sub_proc, sub_f = start_process(
    [PYTHON, "scripts/mqtt/subscribe_verify_mqtt.py", "--max-messages", "20", "--timeout", "60"],
    LOG_DIR / "verify_nb13.log",
)
time.sleep(1)
pub = subprocess.run(
    [PYTHON, "scripts/mqtt/publish_csv_to_mqtt.py", "--limit", "20", "--interval", "0.05"],
    capture_output=True, text=True, timeout=90,
)
print(pub.stdout[:800])
sub_proc.wait(timeout=90)
sub_f.close()
print("\n===== Output subscriber verifikasi =====")
print((LOG_DIR / "verify_nb13.log").read_text(encoding="utf-8")[:1500])

[INFO] Mengirim data dari data/synthetic/room_classification_dataset.csv ke topic iotlab/room01/telemetry
[PUBLISH] {"timestamp": "2026-01-01 00:00:00", "device_id": "room01", "temperature_c": 21.18, "humidity_pct": 53.11, "co2_ppm": 350, "power_watt": 110.65, "status_label": "normal", "is_anomaly": 0}
[PUBLISH] {"timestamp": "2026-01-01 00:10:00", "device_id": "room01", "temperature_c": 21.6, "humidity_pct": 49.71, "co2_ppm": 350, "power_watt": 116.53, "status_label": "normal", "is_anomaly": 0}
[PUBLISH] {"timestamp": "2026-01-01 00:20:00", "device_id": "room01", "temperature_c": 20.75, "humidity_pct": 49.32, "co2_ppm": 350, "power_watt": 95.69, "status_label": "normal", "is_anomaly": 0}
[PUBLISH] {"timestamp": "2026-01-01 00:30:00", "device_id": "room01", "temperature_c": 22.09, "humidit

===== Output subscriber verifikasi =====
[INFO] Terhubung ke broker.
[INFO] Subscribe ke: iotlab/room01/telemetry
[2026-01-01 00:00:00] T=21.18C, H=53.11%, CO2=350, P=110.65W, label=normal, anomaly=

## 2. Publish Multi-Topic (alert warning/fault)

In [4]:
# Ambil 400 baris pertama agar ada beberapa warning/fault di dalamnya.
sub_alert, sub_alert_f = start_process(
    [PYTHON, "scripts/mqtt/subscribe_verify_mqtt.py", "--topic", "iotlab/room01/alert",
     "--timeout", "25"],
    LOG_DIR / "verify_alert_nb13.log",
)
time.sleep(1)
pub = subprocess.run(
    [PYTHON, "scripts/mqtt/publish_multi_topic.py", "--limit", "400", "--interval", "0.01"],
    capture_output=True, text=True, timeout=90,
)
print(pub.stdout[-800:])
sub_alert.wait(timeout=90)
sub_alert_f.close()
print("\n===== Output subscriber alert =====")
print((LOG_DIR / "verify_alert_nb13.log").read_text(encoding="utf-8")[:1200])
broker_proc.terminate(); broker_f.close()
print("[INFO] Broker dihentikan.")

-> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] fault -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] fault -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] warning -> iotlab/room01/alert
[ALERT] fault -> iotlab/room01/alert
[SELESAI] 400 pesan ke iotlab/room01/telemetry, 22 pesan ke iotlab/room01/alert.




===== Output subscriber alert =====
[INFO] Terhubung ke broker.
[INFO] Subscribe ke: iotlab/room01/alert
[2026-01-01 03:50:00] T=25.180546783867907C, H=48.18%, CO2=614, P=172.6430976464329W, label=warning, anomaly=1
[2026-01-01 05:20:00] T=27.06315483956648C, H=46.75%, CO2=716, P=163.11654294786882W, label=warning, anomaly=1
[2026-01-01 07:10:00] T=33.20355310637031C, H=47.17%, CO2=1317, P=277.2267317935645W, label=fault, anomaly=1
[2026-01-01 11:50:00] T=31.87426815884896C, H=49.87%, CO2=948, P=245.99825922224855W, label=warning, anomaly=1
[2026-01-01 14:10:00] T=31.04167799362745C, H=43.16%, CO2=898, P=214.27182576235344W, label=warning, anomaly=1
[2026-01-01 16:10:00] T=30.33189778190201C, H=46.85%, CO2=906, P=216.45544321083344W, label=warning, anomaly=1
[2026-01-01 21:20:00] T=24.507498491240497C, H=49.88%, CO2=578, P=171.7057107074555W, label=warning, anomaly=1
[2026-01-01 22:20:00] T=24.69481952455732C, H=50.73%, CO2=595, P=130.78848771693916W, label=warning, anomaly=1
[2026-01

## Panduan Lengkap dari Buku

_Bagian ini dikutip dari buku PDF (sumber kebenaran praktikum) dan diformat ulang agar mudah dibaca. Kode asli buku tidak diubah; bagian instalasi/persiapan lingkungan diringkas karena sudah dijelaskan di README._

**4. Source code utama: publish CSV ke MQTT**

Buat file scripts/publish_csv_to_mqtt.py:

*(Kode lengkap tersedia pada sel kode notebook ini.)*

Jalankan:

```python
python scripts/publish_csv_to_mqtt.py
```

**5. Subscriber Python untuk memverifikasi**

Buat file scripts/subscribe_verify_mqtt.py:

```python
import json
import paho.mqtt.client as mqtt
BROKER_HOST = "localhost"
BROKER_PORT = 1883
TOPIC = "iotlab/room01/telemetry"
def on_connect(client, userdata, flags, reason_code, properties=None):
    print("Terhubung ke broker.")
    client.subscribe(TOPIC, qos=1)
    print(f"Subscribe ke: {TOPIC}")
def on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode())
        print(
            f"[{payload['timestamp']}] "
            f"T={payload['temperature_c']}C, "
            f"H={payload['humidity_pct']}%, "
            f"CO2={payload['co2_ppm']}, "
            f"P={payload['power_watt']}W, "
            f"label={payload['status_label']}, "
            f"anomaly={payload['is_anomaly']}"
        )
    except Exception as exc:
        print(f"Gagal parse payload: {exc}")
client = mqtt.Client()
client.on_connect = on_connect
client.on_message = on_message
client.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
client.loop_forever()
```

**7. Versi publish multi-topic**

Contoh pengiriman berdasarkan label:

```python
from pathlib import Path
import json
import time
import pandas as pd
import paho.mqtt.client as mqtt
BROKER_HOST = "localhost"
BROKER_PORT = 1883
DATA_PATH = Path("data/synthetic/room_classification_dataset.csv")
client = mqtt.Client(client_id="multi-topic-publisher")
client.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
df = pd.read_csv(DATA_PATH)
for _, row in df.iterrows():
    payload = row.to_dict()
    payload_json = json.dumps(payload)
    telemetry_topic = "iotlab/room01/telemetry"
    client.publish(telemetry_topic, payload_json, qos=1)
    if row["status_label"] in ["warning", "fault"]:
        alert_topic = "iotlab/room01/alert"
        client.publish(alert_topic, payload_json, qos=1)
    print(payload_json)
    time.sleep(0.3)
client.disconnect()
```

## Tantangan dan Variasi dari Buku

**6. Variasi realistis yang bisa dicoba**

Agar lebih menantang, pembaca bisa mencoba:

- publish hanya data warning dan fault ke topik khusus

iotlab/room01/alert

- publish multi-device seperti room01, room02, lab01
- memperlambat atau mempercepat simulasi
- menambahkan battery_pct, location, atau sensor_status
- mengirim data energi atau getaran ke topik lain

## Solusi Variasi (Tambahan di Luar Buku)

_Bagian ini BUKAN dari buku. Variasi pada bagian sebelumnya dikerjakan di sini; kode buku tetap apa adanya. Variasi "publish hanya warning/fault ke topik alert" sudah dijalankan pada sel sebelumnya (`scripts/mqtt/publish_multi_topic.py`)._

In [5]:
# [SOLUSI 2-4] Publish multi-device (room01, room02, lab01) dengan battery_pct,
# location, sensor_status, dan interval simulasi yang dipercepat.
import json
import random
import time

import pandas as pd
import paho.mqtt.client as mqtt

df_sol = pd.read_csv("data/synthetic/room_classification_dataset.csv").head(30)
perangkat = {"room01": "Lab 1", "room02": "Lab 1", "lab01": "Lab 2"}

sol_broker2, sol_broker2_f = start_process(
    [PYTHON, "scripts/mqtt/run_broker.py"], LOG_DIR / "broker_solusi_nb13.log")
time.sleep(3)

pub = mqtt.Client(client_id="solusi-multidevice-pub")
pub.connect("localhost", 1883, keepalive=60)
for dev, lokasi in perangkat.items():
    for _, row in df_sol.head(3).iterrows():
        payload = row.to_dict()
        payload["device_id"] = dev
        payload["location"] = lokasi                 # (4) field location
        payload["battery_pct"] = random.randint(20, 100)  # (4) field battery_pct
        payload["sensor_status"] = "ok"              # (4) field sensor_status
        pub.publish(f"iotlab/{dev}/telemetry", json.dumps(payload, default=str), qos=1)
        time.sleep(0.02)                             # (3) interval dipercepat
pub.disconnect()
print("[SUKSES] Telemetri multi-device terkirim untuk room01, room02, lab01.")

# [SOLUSI 5] Data energi dikirim ke topik lain.
df_energy = pd.read_csv("data/synthetic/energy_30days.csv").head(5)
pub = mqtt.Client(client_id="solusi-energy-pub")
pub.connect("localhost", 1883, keepalive=60)
for _, row in df_energy.iterrows():
    pub.publish("iotlab/energy/telemetry", json.dumps(row.to_dict(), default=str), qos=1)
pub.disconnect()
print("[SUKSES] 5 baris data energi terkirim ke topik iotlab/energy/telemetry.")

sol_broker2.terminate(); sol_broker2_f.close()
print("[INFO] Broker solusi dihentikan.")

[SUKSES] Telemetri multi-device terkirim untuk room01, room02, lab01.
[SUKSES] 5 baris data energi terkirim ke topik iotlab/energy/telemetry.
[INFO] Broker solusi dihentikan.


## Penutup

Dataset CSV kini mengalir lewat MQTT seperti sensor sungguhan: telemetri penuh di satu
topik, dan alert khusus untuk kondisi warning/fault. Pola ini menjadi dasar pipeline
end-to-end pada Hands-on 30.